# Chapter 15 — Supplement 7: the reasoning record

*Typed claims with evidence, not free-form chain-of-thought.*

Companion to `15_capstone.ipynb`. Supplements 1-6 opened each tool and the gate stack. This one opens the agent's **reasoning record**: the structured scratchpad the complaint agent keeps as it works, and the evidence check it runs before it lets the model draft a reply. It is the Chapter 3 discipline -- *a model's reasoning is a working artifact, not evidence* -- made operational in the capstone. All code calls the shipped `glassloop.reasoning` and `glassloop.capstone` implementations.

## The idea

A language model's free-form 'thinking' is useful to the model and worthless as proof to the rest of the system. Chapter 3 replaces it with a `Scratchpad` of **typed entries**, each carrying:

- a **kind** -- `claim`, `observation`, `assumption` or `question`;
- a **trust level** -- `low < medium < high < highest`;
- an **evidence pointer** -- required for anything above `low` trust;
- a **source** -- required for every observation.

The invariants are enforced *at construction* (Pydantic), so an entry that should cite something but does not cannot be built. The capstone agent records each tool result as one of these entries and, before drafting, asserts that every claim is evidenced.

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

root = Path('.')
if not (root / 'data').exists():
    root = Path('..')
assert (root / 'data').exists(), 'run this notebook from the repo root or notebooks/'
print('repo root:', root.resolve())

## 1. The `Scratchpad` primitive, directly

First the Chapter 3 class itself. We add one of each kind of entry and render the table the agent and the audit log both read.

In [ ]:
from glassloop.reasoning import Scratchpad, Entry, EntryType, TrustLevel
from pydantic import ValidationError

pad = Scratchpad()
pad.add_observation('two charges posted on 2026-05-01', source='ticket-1042',
                    evidence='ticket-1042', trust=TrustLevel.MEDIUM)
pad.add_claim('the overdraft fee is $35', evidence='overdraft.txt', trust=TrustLevel.HIGH)
pad.add_assumption('the customer was charged twice in the same day')
pad.add_question('was a goodwill reversal already issued this year?')
print(pad.render_table(as_string=True))

Each entry is typed and trust-rated, and the assumption is labelled as an assumption -- not silently promoted to fact. The invariants that make those trust levels mean something fire *at construction*, not downstream:

In [ ]:
try:
    Entry(kind=EntryType.OBSERVATION, text='classified as complaint')  # no source
except ValidationError as e:
    print('observation without a source ->', e.errors()[0]['msg'])

try:
    Entry(kind=EntryType.CLAIM, text='fee is $35', trust=TrustLevel.HIGH)  # no evidence
except ValidationError as e:
    print('high-trust claim without evidence ->', e.errors()[0]['msg'])

## 2. How the capstone fills the scratchpad

The `ComplaintAgent` reconstructs the record as a **pure function of the trajectory** -- each tool result becomes one or more entries -- so the same run always yields the same record and it replays from the audit log without rerunning the agent. The mapping follows the trust hierarchy:

| Tool result | Entry | Trust | Evidence |
| --- | --- | --- | --- |
| `classify_complaint` label | observation | medium | classifier confidence |
| `extract_facts` issue | claim | medium | the extractor over the message |
| urgency / sentiment (inferred) | **assumption** | low | none -- by design |
| `search_policy` hit | observation | high | the policy id (a cited source) |
| `flag_regulatory` flag | claim | **highest** | the GMS regulatory graph |

Inferred-but-uncited fields are recorded as assumptions, never claims, so they can never pass an evidence check they have not earned.

## 3. Inspect a real case's record

We build the shipped harness and run one complaint. The compiled output carries the record under `final_output['reasoning']`.

In [ ]:
from glassloop.capstone import build_complaint_harness
from glassloop.core import Budget, BudgetTracker, TaskSpec

harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())

def reasoning_for(cid):
    case = next(c for c in cases if c['id'] == cid)
    task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(task, max_steps=16,
                       budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    return case, traj

case, traj = reasoning_for('case-003')
print('message:', case['message'])
print()
for e in traj.final_state.final_output['reasoning']:
    ev = e['evidence'] or '(none)'
    print(f"  {e['kind']:<11} {e['trust']:<7} {e['text']:<40} [{ev}]")

Read it against the hierarchy. Every **claim** carries an evidence pointer: the two regulatory claims, derived by the GMS graph from evidence in the message, sit at the highest trust; the issue claim, grounded only in the extractor's reading of the text, sits at medium. The model's guess at urgency and sentiment is an **assumption** with no evidence -- precisely so it cannot be mistaken for fact.

## 4. The evidence gate has teeth

Before the agent lets `draft_response` run, it calls `assert_all_claims_have_evidence()` on the scratchpad. On a real case every claim is evidenced, so it passes. To see the gate bite, build a scratchpad with an unsupported claim -- the same call an ungrounded reasoning step would make:

In [ ]:
good = Scratchpad()
good.add_claim('issue is credit_card_issue', evidence='extract_facts', trust=TrustLevel.MEDIUM)
good.add_claim('regulation implicated: Reg_E', evidence='GMS graph', trust=TrustLevel.HIGHEST)
good.assert_all_claims_have_evidence()
print('grounded scratchpad: assertion passed')

bad = Scratchpad()
bad.add_claim('issue is credit_card_issue', evidence='extract_facts', trust=TrustLevel.MEDIUM)
bad.add_claim('the customer is owed a full refund')   # no evidence pointer
print('unsupported claims:', [e.text for e in bad.unsupported_claims()])
try:
    bad.assert_all_claims_have_evidence()
except AssertionError as e:
    print('blocked before drafting ->', e)

In the agent that `AssertionError` path returns an `Escalate`: an ungrounded claim sends the case to a human instead of reaching the customer. The check is structural -- a property of the scratchpad -- not a second model's opinion.

## 5. The record adapts to the case

A case that fires a regulation carries `highest`-trust regulatory claims; one that does not, will not. Compare case-003 (an unauthorized-charge dispute, flags Reg_E/Reg_Z) with case-020 (a double charge, no regulatory flag):

In [ ]:
for cid in ['case-003', 'case-020']:
    case, traj = reasoning_for(cid)
    rows = traj.final_state.final_output['reasoning']
    claims = [r for r in rows if r['kind'] == 'claim']
    print(f"{cid}: {len(rows)} entries, {len(claims)} claims")
    for r in claims:
        print(f"    [{r['trust']:<7}] {r['text']}  <- {r['evidence']}")
    print()

case-003 carries two `highest`-trust regulatory claims; case-020 carries only the medium-trust issue claim, because the guard found no regulation its evidence supports. The record reflects exactly what the run established, nothing more.

## 6. The record is reproducible content

Because the record is a pure function of the trajectory and the models decode greedily, the **typed content** is identical across runs (only the per-entry ids, freshly minted on each construction, differ):

In [ ]:
def signature(rows):
    return [(r['kind'], r['trust'], r['text'], r['evidence']) for r in rows]

_, t1 = reasoning_for('case-003')
_, t2 = reasoning_for('case-003')
s1 = signature(t1.final_state.final_output['reasoning'])
s2 = signature(t2.final_state.final_output['reasoning'])
print('typed content identical across runs:', s1 == s2)

## Summary

- The capstone keeps a Chapter 3 `Scratchpad`: every recorded fact is a typed entry with a trust level and, above the lowest trust, an evidence pointer -- enforced at construction.
- Model outputs enter at the trust the hierarchy allows (a classifier label is a medium observation; a GMS-confirmed regulation is a highest-trust claim); inferred fields are assumptions, not claims.
- Before drafting, the agent asserts every claim is evidenced; an ungrounded claim escalates to a human.
- The record is a pure function of the trajectory, so its content replays from the audit log without rerunning the agent.

This closes the loop opened in Chapter 3: reasoning is recorded as checkable, typed claims, never as free-form prose the rest of the system has to trust.